In [ ]:
# ================================================================
# EXAMPLE 6:
# 1D STEADY CONVECTION-DIFFUSION USING FDM
#
# Comparison of:
#   1. Central Difference Scheme (CDS)
#   2. Upwind Difference Scheme (UDS)
#   3. Analytical Solution
# ================================================================
#
# Governing equation:
#
#              u dT/dx = alpha d²T/dx²
#
# Boundary conditions:
#
#              T(0) = T_left
#              T(L) = T_right
#
# ================================================================


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ================================================================
# 1. USER INPUTS
# ================================================================
#
# CHANGE ONLY THESE VALUES FOR DIFFERENT PROBLEMS
# ================================================================

L = 1.0                 # Domain length (m)

N = 4                   # Total number of grid nodes

u = -0.25                # Fluid velocity (m/s)

alpha = 0.01            # Thermal diffusivity (m²/s)

T_left = 100.0          # Left boundary temperature (°C)

T_right = 0.0           # Right boundary temperature (°C)


# ================================================================
# 2. BASIC INPUT CHECKS
# ================================================================

if N < 3:
    raise ValueError("N must be at least 3.")

if L <= 0:
    raise ValueError("L must be positive.")

if alpha <= 0:
    raise ValueError("alpha must be positive.")


# ================================================================
# 3. COMPUTATIONAL GRID
# ================================================================

dx = L / (N - 1)

x = np.linspace(0, L, N)

num_unknowns = N - 2


print("================================================")
print("GRID INFORMATION")
print("================================================")

print("Domain length L          =", L, "m")
print("Total number of nodes N  =", N)
print("Grid spacing dx          =", dx, "m")
print("Node positions           =", x)
print("Number of unknowns       =", num_unknowns)

print()


# ================================================================
# 4. FLOW DIRECTION
# ================================================================

print("================================================")
print("FLOW INFORMATION")
print("================================================")

print("Velocity u =", u, "m/s")


if u > 0:

    print("Flow direction: LEFT ---> RIGHT")

    print(
        "For Upwind Scheme: use BACKWARD difference"
    )


elif u < 0:

    print("Flow direction: RIGHT ---> LEFT")

    print(
        "For Upwind Scheme: use FORWARD difference"
    )


else:

    print("u = 0: No convection, only diffusion")


print()


# ================================================================
# 5. PECLET NUMBERS
# ================================================================
#
# Domain Peclet number:
#
#              Pe_L = u L / alpha
#
#
# Grid Peclet number:
#
#              Pe_dx = u dx / alpha
#
# ================================================================

Pe_L = u * L / alpha

Pe_dx = u * dx / alpha


print("================================================")
print("PECLET NUMBER INFORMATION")
print("================================================")

print("Domain Peclet number Pe_L  =", Pe_L)

print("Grid Peclet number Pe_dx   =", Pe_dx)

print()


# ================================================================
# 6. CENTRAL DIFFERENCE BOUNDEDNESS CHECK
# ================================================================
#
# For the 1D Central Difference Scheme:
#
#              |Pe_dx| <= 2
#
# is the usual boundedness requirement.
#
# ================================================================

if abs(Pe_dx) <= 2:

    print("CDS CHECK: |Pe_dx| <= 2")

    print("Central Difference Scheme is expected to be bounded.")


else:

    print("CDS WARNING: |Pe_dx| > 2")

    print(
        "Central Difference Scheme may produce "
        "nonphysical oscillations."
    )


print()


# ================================================================
# 7. CENTRAL DIFFERENCE SCHEME
# ================================================================
#
# For convection:
#
#       dT/dx
#
#       ~ (T(i+1) - T(i-1)) / (2 dx)
#
#
# For diffusion:
#
#       d²T/dx²
#
#       ~ (T(i-1) - 2T(i) + T(i+1)) / dx²
#
#
# Resulting positive-diagonal equation:
#
#   2T(i)
#
#   - (1 + Pe_dx/2) T(i-1)
#
#   - (1 - Pe_dx/2) T(i+1)
#
#   = 0
#
# ================================================================


# CDS coefficients

aW_cds = -(1.0 + Pe_dx / 2.0)

aP_cds = 2.0

aE_cds = -(1.0 - Pe_dx / 2.0)


# Initialize CDS matrix

A_cds = np.zeros(
    (num_unknowns, num_unknowns)
)

b_cds = np.zeros(
    num_unknowns
)


# ================================================================
# 8. ASSEMBLE CDS MATRIX
# ================================================================

for j in range(num_unknowns):

    # Physical node number
    i = j + 1


    # Central coefficient
    A_cds[j, j] = aP_cds


    # ------------------------------------------------------------
    # West / left neighbor
    # ------------------------------------------------------------

    if i - 1 == 0:

        # Known left boundary
        b_cds[j] -= aW_cds * T_left

    else:

        A_cds[j, j - 1] = aW_cds


    # ------------------------------------------------------------
    # East / right neighbor
    # ------------------------------------------------------------

    if i + 1 == N - 1:

        # Known right boundary
        b_cds[j] -= aE_cds * T_right

    else:

        A_cds[j, j + 1] = aE_cds


# ================================================================
# 9. SOLVE CDS SYSTEM
# ================================================================

T_cds_interior = np.linalg.solve(
    A_cds,
    b_cds
)


T_cds = np.zeros(N)

T_cds[0] = T_left

T_cds[-1] = T_right

T_cds[1:-1] = T_cds_interior


# ================================================================
# 10. UPWIND DIFFERENCE SCHEME
# ================================================================
#
# IMPORTANT:
#
# Upwind depends on FLOW DIRECTION.
#
#
# If u > 0:
#
#       flow is left ---> right
#
#       dT/dx ~ (T_i - T_(i-1)) / dx
#
#       BACKWARD difference
#
#
# If u < 0:
#
#       flow is right ---> left
#
#       dT/dx ~ (T_(i+1) - T_i) / dx
#
#       FORWARD difference
#
# ================================================================


# ================================================================
# CASE A: u > 0
# ================================================================

if u > 0:

    # Equation:
    #
    # (Pe + 2) T_i
    #
    # - (Pe + 1) T_(i-1)
    #
    # - T_(i+1)
    #
    # = 0

    aW_up = -(Pe_dx + 1.0)

    aP_up = Pe_dx + 2.0

    aE_up = -1.0


# ================================================================
# CASE B: u < 0
# ================================================================

elif u < 0:

    # Forward upwind approximation
    #
    # Equation:
    #
    # (2 - Pe) T_i
    #
    # - T_(i-1)
    #
    # - (1 - Pe) T_(i+1)
    #
    # = 0

    aW_up = -1.0

    aP_up = 2.0 - Pe_dx

    aE_up = Pe_dx - 1.0


# ================================================================
# CASE C: u = 0
# ================================================================

else:

    # Pure diffusion

    aW_up = -1.0

    aP_up = 2.0

    aE_up = -1.0


# ================================================================
# 11. ASSEMBLE UPWIND MATRIX
# ================================================================

A_up = np.zeros(
    (num_unknowns, num_unknowns)
)

b_up = np.zeros(
    num_unknowns
)


for j in range(num_unknowns):

    i = j + 1


    # Central coefficient

    A_up[j, j] = aP_up


    # ------------------------------------------------------------
    # West / left neighbor
    # ------------------------------------------------------------

    if i - 1 == 0:

        b_up[j] -= aW_up * T_left

    else:

        A_up[j, j - 1] = aW_up


    # ------------------------------------------------------------
    # East / right neighbor
    # ------------------------------------------------------------

    if i + 1 == N - 1:

        b_up[j] -= aE_up * T_right

    else:

        A_up[j, j + 1] = aE_up


# ================================================================
# 12. SOLVE UPWIND SYSTEM
# ================================================================

T_up_interior = np.linalg.solve(
    A_up,
    b_up
)


T_upwind = np.zeros(N)

T_upwind[0] = T_left

T_upwind[-1] = T_right

T_upwind[1:-1] = T_up_interior


# ================================================================
# 13. DISPLAY CDS MATRIX
# ================================================================

if N <= 12:

    print("================================================")
    print("CENTRAL DIFFERENCE MATRIX A")
    print("================================================")

    print(A_cds)

    print()


    print("CENTRAL DIFFERENCE RHS b")

    print(b_cds)

    print()


# ================================================================
# 14. DISPLAY UPWIND MATRIX
# ================================================================

if N <= 12:

    print("================================================")
    print("UPWIND DIFFERENCE MATRIX A")
    print("================================================")

    print(A_up)

    print()


    print("UPWIND RHS b")

    print(b_up)

    print()


# ================================================================
# 15. ANALYTICAL SOLUTION
# ================================================================
#
# Exact solution:
#
# T(x)
#
# = T_left
#
# + (T_right - T_left)
#
#   [ exp(u x / alpha) - 1 ]
#   -------------------------
#   [ exp(u L / alpha) - 1 ]
#
#
# If u = 0, the solution becomes linear.
#
# ================================================================


if np.isclose(u, 0.0):

    T_exact = (
        T_left
        + (T_right - T_left)
        * x / L
    )


else:

    T_exact = (

        T_left

        + (T_right - T_left)

        * np.expm1(
            u * x / alpha
        )

        / np.expm1(
            u * L / alpha
        )

    )


# ================================================================
# 16. ABSOLUTE ERRORS
# ================================================================

error_cds = np.abs(
    T_cds - T_exact
)

error_upwind = np.abs(
    T_upwind - T_exact
)


# ================================================================
# 17. RELATIVE ERRORS
# ================================================================
#
# Relative error is not calculated when the exact value is zero.
#
# ================================================================

relative_cds = np.full(
    N,
    np.nan
)

relative_upwind = np.full(
    N,
    np.nan
)


mask = np.abs(T_exact) > 1.0e-12


relative_cds[mask] = (

    error_cds[mask]
    / np.abs(T_exact[mask])
    * 100.0

)


relative_upwind[mask] = (

    error_upwind[mask]
    / np.abs(T_exact[mask])
    * 100.0

)


# ================================================================
# 18. RESULTS TABLE
# ================================================================

results = pd.DataFrame({

    "Node":
        np.arange(N),

    "x (m)":
        np.round(x, 5),

    "Analytical T (°C)":
        np.round(T_exact, 5),

    "CDS T (°C)":
        np.round(T_cds, 5),

    "Upwind T (°C)":
        np.round(T_upwind, 5),

    "CDS Abs. Error":
        np.round(error_cds, 5),

    "Upwind Abs. Error":
        np.round(error_upwind, 5),

    "CDS Rel. Error (%)":
        np.round(relative_cds, 4),

    "Upwind Rel. Error (%)":
        np.round(relative_upwind, 4)

})


print("================================================")
print("CDS, UPWIND AND ANALYTICAL COMPARISON")
print("================================================")

display(results)


# ================================================================
# 19. PRINT COMPLETE TEMPERATURE VECTORS
# ================================================================

print("================================================")
print("TEMPERATURE VECTORS")
print("================================================")

print(
    "Analytical :",
    np.round(T_exact, 5)
)

print(
    "CDS        :",
    np.round(T_cds, 5)
)

print(
    "Upwind     :",
    np.round(T_upwind, 5)
)

print()


# ================================================================
# 20. CHECK WHETHER NUMERICAL SOLUTIONS ARE BOUNDED
# ================================================================

T_min = min(
    T_left,
    T_right
)

T_max = max(
    T_left,
    T_right
)


cds_bounded = np.all(
    (T_cds >= T_min - 1e-10)
    &
    (T_cds <= T_max + 1e-10)
)


upwind_bounded = np.all(
    (T_upwind >= T_min - 1e-10)
    &
    (T_upwind <= T_max + 1e-10)
)


print("================================================")
print("BOUNDEDNESS CHECK")
print("================================================")

print(
    "CDS bounded between boundary temperatures?   ",
    cds_bounded
)

print(
    "Upwind bounded between boundary temperatures?",
    upwind_bounded
)

print()


# ================================================================
# 21. SMOOTH ANALYTICAL CURVE
# ================================================================

x_exact = np.linspace(
    0,
    L,
    400
)


if np.isclose(u, 0.0):

    T_exact_plot = (

        T_left

        + (T_right - T_left)
        * x_exact / L

    )


else:

    T_exact_plot = (

        T_left

        + (T_right - T_left)

        * np.expm1(
            u * x_exact / alpha
        )

        / np.expm1(
            u * L / alpha
        )

    )


# ================================================================
# 22. PLOT:
# CDS vs UPWIND vs ANALYTICAL
# ================================================================

plt.figure(
    figsize=(9, 6)
)


# ------------------------------------------------
# Analytical continuous solution
# ------------------------------------------------

plt.plot(

    x_exact,

    T_exact_plot,

    linewidth=2,

    label="Analytical Solution"

)


# ------------------------------------------------
# CDS discrete nodal solution
# ------------------------------------------------

plt.scatter(

    x,

    T_cds,

    s=80,

    marker="o",

    label="Central Difference (CDS)"

)


# ------------------------------------------------
# Upwind discrete nodal solution
# ------------------------------------------------

plt.scatter(

    x,

    T_upwind,

    s=80,

    marker="s",

    label="Upwind Difference"

)


plt.xlabel(
    "Position x (m)"
)

plt.ylabel(
    "Temperature T (°C)"
)


plt.title(

    "1D Steady Convection-Diffusion\n"

    f"Pe_dx = {Pe_dx:.2f}, "
    f"u = {u} m/s, "
    f"N = {N}"

)


plt.grid(True)

plt.legend()

plt.tight_layout()

plt.show()


# ================================================================
# 23. FINAL INTERPRETATION
# ================================================================

print("================================================")
print("NUMERICAL INTERPRETATION")
print("================================================")


if abs(Pe_dx) <= 2:

    print(
        "|Pe_dx| <= 2:"
    )

    print(
        "CDS is within its usual boundedness range."
    )


else:

    print(
        "|Pe_dx| > 2:"
    )

    print(
        "CDS may produce oscillations because "
        "convection is too strong relative to diffusion "
        "on this grid."
    )


print()


if u > 0:

    print(
        "Because u > 0, flow is left to right."
    )

    print(
        "Therefore Upwind uses the BACKWARD difference:"
    )

    print(
        "dT/dx ≈ (T_i - T_(i-1))/dx"
    )


elif u < 0:

    print(
        "Because u < 0, flow is right to left."
    )

    print(
        "Therefore Upwind uses the FORWARD difference:"
    )

    print(
        "dT/dx ≈ (T_(i+1) - T_i)/dx"
    )


print()